# COMP8410 — Roblox Cube 3D v0.5 Hands-On Notebook
## Clean Google Colab / T4 version

This notebook fixes the dependency conflict in the previous `Copy_of_cube3d` notebook.

**What was wrong:** the previous notebook upgraded `huggingface-hub` to 2.x, while the installed Transformers build required `huggingface-hub < 2.0`. The Cube model weights themselves downloaded successfully.

### Instructions
1. In Colab choose **Runtime → Disconnect and delete runtime**.
2. Open this notebook.
3. Choose **Runtime → Change runtime type → T4 GPU**.
4. Run the cells in order.
5. Do **not** add `pip install -U huggingface_hub`.
6. This notebook uses the normal Cube engine (not `--fast-inference`) and starts at `resolution-base 4.0` to stay within a free T4's memory budget.

The notebook generates three meshes, measures them, visualises them, and packages the evidence for your assignment.

In [ ]:
# 1. Confirm GPU
import subprocess, sys, os

subprocess.run(["nvidia-smi"], check=False)
print("\nPython:", sys.version)

In [ ]:
# 2. Fresh clone of the official Roblox Cube repository
import os, shutil, subprocess

REPO = "/content/cube"
os.chdir("/content")

if os.path.exists(REPO):
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "-q", "https://github.com/Roblox/cube.git", REPO],
    check=True
)
os.chdir(REPO)
print("Cloned:", REPO)

In [ ]:
# 3. Install a compatible dependency stack.
# IMPORTANT: huggingface-hub stays BELOW 2.0.

import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.16.1",
    "huggingface-hub==1.33.0"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e", "."
])

print("Cube and compatible Hugging Face dependencies installed.")

In [ ]:
# 4. Verify the exact versions BEFORE downloading / inference.

import transformers
import huggingface_hub
import torch
import trimesh
import numpy as np

print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

assert tuple(map(int, huggingface_hub.__version__.split(".")[:1])) < (2,), (
    "huggingface_hub must remain below 2.0 for this Transformers build."
)
assert torch.cuda.is_available(), "Enable a T4 GPU before continuing."

In [ ]:
# 5. Download Roblox Cube 3D v0.5 weights using the Python API.
# This avoids the changing Hugging Face CLI command.

from huggingface_hub import snapshot_download
import os

WEIGHTS = "/content/cube/model_weights"

snapshot_download(
    repo_id="Roblox/cube3d-v0.5",
    local_dir=WEIGHTS
)

gpt_path = os.path.join(WEIGHTS, "shape_gpt.safetensors")
shape_path = os.path.join(WEIGHTS, "shape_tokenizer.safetensors")

assert os.path.exists(gpt_path)
assert os.path.exists(shape_path)

print("\nDownloaded successfully:")
print("GPT:", gpt_path, round(os.path.getsize(gpt_path)/1024**3, 2), "GB")
print("Tokenizer:", shape_path, round(os.path.getsize(shape_path)/1024**3, 2), "GB")

## Generation experiment

We use three prompts with different geometry and aspect ratios:

1. **Tall pagoda** — vertically elongated architecture.
2. **Retro sports car** — horizontally elongated mechanical object.
3. **Ornate wooden throne** — detailed furniture with a tall back.

The bounding boxes intentionally differ so you can discuss **bounding-box conditioning** in Cube 3D v0.5.

The script uses the normal `Engine`, **not `EngineFast`**, because a T4 has only 16 GB VRAM. It starts at `resolution-base 4.0`. If a generation fails specifically because of CUDA memory, the cell retries once at 3.5.

In [ ]:
# 6. Generate three Cube 3D meshes and record wall-clock time.

import os, subprocess, time, json

os.chdir("/content/cube")
OUTPUT_ROOT = "/content/cube/assignment_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

experiments = [
    {
        "name": "pagoda",
        "prompt": "A tall pagoda",
        "bbox": [1.0, 2.0, 1.5],
    },
    {
        "name": "car",
        "prompt": "A retro sports car",
        "bbox": [2.0, 1.0, 1.0],
    },
    {
        "name": "throne",
        "prompt": "An ornate wooden throne",
        "bbox": [1.0, 1.5, 1.0],
    },
]

generation_records = []

def run_generation(exp, resolution):
    out_dir = os.path.join(OUTPUT_ROOT, exp["name"])
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        sys.executable, "-m", "cube3d.generate",
        "--gpt-ckpt-path", gpt_path,
        "--shape-ckpt-path", shape_path,
        "--prompt", exp["prompt"],
        "--bounding-box-xyz",
        str(exp["bbox"][0]), str(exp["bbox"][1]), str(exp["bbox"][2]),
        "--resolution-base", str(resolution),
        "--output-dir", out_dir,
    ]

    print("\n" + "="*75)
    print("Generating:", exp["prompt"])
    print("Bounding box:", exp["bbox"])
    print("Resolution base:", resolution)

    start = time.perf_counter()
    proc = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    elapsed = time.perf_counter() - start

    print(proc.stdout[-5000:])

    return proc.returncode, proc.stdout, elapsed, out_dir

for exp in experiments:
    code_rc, log, elapsed, out_dir = run_generation(exp, 4.0)
    used_resolution = 4.0

    if code_rc != 0 and "out of memory" in log.lower():
        print("CUDA OOM detected. Retrying once at resolution-base 3.5...")
        code_rc, log, elapsed, out_dir = run_generation(exp, 3.5)
        used_resolution = 3.5

    if code_rc != 0:
        raise RuntimeError(
            f"Generation failed for {exp['name']}. "
            "Copy the printed error back to ChatGPT."
        )

    obj_path = os.path.join(out_dir, "output.obj")
    if not os.path.exists(obj_path):
        raise FileNotFoundError(f"Expected OBJ was not created: {obj_path}")

    generation_records.append({
        "name": exp["name"],
        "prompt": exp["prompt"],
        "bbox_x": exp["bbox"][0],
        "bbox_y": exp["bbox"][1],
        "bbox_z": exp["bbox"][2],
        "resolution_base": used_resolution,
        "generation_seconds": round(elapsed, 3),
        "obj_path": obj_path,
    })

print("\nSUCCESS: all requested meshes were generated.")

In [ ]:
# 7. Analyse each generated mesh quantitatively.

import trimesh
import numpy as np

records = []

for rec in generation_records:
    mesh = trimesh.load(rec["obj_path"], force="mesh")

    components = mesh.split(only_watertight=False)

    row = dict(rec)
    row.update({
        "vertices": int(len(mesh.vertices)),
        "faces": int(len(mesh.faces)),
        "watertight": bool(mesh.is_watertight),
        "connected_components": int(len(components)),
        "extent_x": round(float(mesh.extents[0]), 5),
        "extent_y": round(float(mesh.extents[1]), 5),
        "extent_z": round(float(mesh.extents[2]), 5),
        "surface_area": round(float(mesh.area), 5),
    })
    records.append(row)

    print("\n", rec["name"].upper())
    print("Prompt:", rec["prompt"])
    print("Generation time:", rec["generation_seconds"], "s")
    print("Vertices:", row["vertices"])
    print("Faces:", row["faces"])
    print("Watertight:", row["watertight"])
    print("Connected components:", row["connected_components"])
    print("Extents:", mesh.extents)

In [ ]:
# 8. Save exact experiment measurements to CSV + JSON.

import csv, json, os

csv_path = os.path.join(OUTPUT_ROOT, "cube3d_results.csv")
json_path = os.path.join(OUTPUT_ROOT, "cube3d_results.json")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print("Saved:", csv_path)
print("Saved:", json_path)

In [ ]:
# 9. Compact evidence table — take a screenshot of this cell.

print("-" * 120)
print(
    f"{'Object':<12}{'Resolution':>12}{'Time(s)':>12}"
    f"{'Vertices':>14}{'Faces':>14}{'Watertight':>13}{'Components':>13}"
)
print("-" * 120)

for r in records:
    print(
        f"{r['name']:<12}"
        f"{r['resolution_base']:>12.1f}"
        f"{r['generation_seconds']:>12.2f}"
        f"{r['vertices']:>14}"
        f"{r['faces']:>14}"
        f"{str(r['watertight']):>13}"
        f"{r['connected_components']:>13}"
    )

In [ ]:
# 10. Interactive 3D visualisation — rotate each mesh and take screenshots.

import plotly.graph_objects as go

for r in records:
    mesh = trimesh.load(r["obj_path"], force="mesh")
    v = np.asarray(mesh.vertices)
    f = np.asarray(mesh.faces)

    fig = go.Figure(
        data=[
            go.Mesh3d(
                x=v[:,0], y=v[:,1], z=v[:,2],
                i=f[:,0], j=f[:,1], k=f[:,2],
                intensity=v[:,2],
                colorscale="Viridis",
                showscale=False,
                flatshading=False,
            )
        ]
    )

    fig.update_layout(
        title=f"Cube 3D v0.5 — {r['prompt']}",
        scene_aspectmode="data",
        width=820,
        height=650,
        margin=dict(l=0, r=0, b=0, t=45),
    )
    fig.show()

## What to capture for the report

Take screenshots of:

- the **T4 GPU / runtime** cell;
- the **successful model-weight download**;
- at least **two generated meshes** from different prompts;
- the **quantitative evidence table**;
- optionally a mesh where Cube 3D produces disconnected or malformed geometry.

### What to discuss

For each generated asset, discuss:
- how well the mesh follows the text prompt;
- whether the requested bounding-box proportions are reflected in the shape;
- visible geometric artifacts;
- disconnected components;
- topology / watertightness;
- generation time;
- how resolution-base affects quality and compute.

**Important for your assignment:** Cube 3D is your selected case-study system. Your assignment brief separately asks for a lightweight **open-source counterpart** in Section 3.2, and specifically suggests TripoSR or InstantMesh for 3D generation. Therefore, treat these Cube results as additional supporting evidence; the safest rubric-aligned hands-on experiment remains the TripoSR/InstantMesh counterpart.

In [ ]:
# 11. Package all generated meshes and measurements.

import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/COMP8410_Cube3D_evidence",
    "zip",
    OUTPUT_ROOT
)

print("Created:", zip_path)
files.download(zip_path)